# Feature Engineering II: Behavioural Features, Selection and Leakage-Safe Pipelines

Notebook for **Chapter 6** of the *ds* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ds/ch06.html

Tip: change a number, predict the output, then run the cell again.

## 6.1 RFM and Rolling-Window Features

In [ ]:
def build_features(c, o, T):
    """One row per ACTIVE customer at T. Uses only orders with order_ts < T. Label from T..T+60d."""
    T = pd.Timestamp(T)
    base = make_label_table(c, o, pred_date=T)               # customers + churn label
    past = o[o.order_ts < T].copy()                           # <-- the leakage guard
    past["age_d"] = (T - past.order_ts).dt.days               # 1 = yesterday
    f = pd.DataFrame(index=base.customer_id)
    g = past.groupby("customer_id")
    f["recency_days"] = g.age_d.min()                         # R
    f["frequency_all"] = g.size()                             # F
    f["monetary_all"] = g.amount.sum()                        # M
    for w in (7, 30, 90):                                     # rolling windows
        s = past[past.age_d <= w].groupby("customer_id")
        f[f"orders_{w}d"] = s.size(); f[f"spend_{w}d"] = s.amount.sum()
    s = past[(past.age_d > 30) & (past.age_d <= 60)].groupby("customer_id")     # previous windows
    f["orders_30_60d"] = s.size(); f["spend_30_60d"] = s.amount.sum()
    s = past[(past.age_d > 60) & (past.age_d <= 90)].groupby("customer_id")
    f["orders_60_90d"] = s.size(); f["spend_60_90d"] = s.amount.sum()
    f = f.reindex(base.customer_id).fillna({k: 0 for k in f.columns if k != "recency_days"})
    f["avg_order_90d"] = np.where(f.orders_90d > 0, f.spend_90d / f.orders_90d.clip(lower=1), np.nan)
    f["ratio_7_30"] = f.orders_7d / (f.orders_30d + 1)
    f["trend_30_vs_prev60"] = (f.orders_30d + 1) / ((f.orders_30_60d + f.orders_60_90d) / 2 + 1)
    f["velocity_spend"] = f.spend_30d - f.spend_30_60d
    f["accel_spend"] = (f.spend_30d - f.spend_30_60d) - (f.spend_30_60d - f.spend_60_90d)
    # gaps between consecutive orders in the last 180 days
    p2 = past[past.age_d <= 180].sort_values(["customer_id", "order_ts"])
    p2["gap"] = p2.groupby("customer_id").order_ts.diff().dt.days
    gg = p2.groupby("customer_id").gap.agg(["mean", "max", "std"]); gg.columns = ["gap_mean", "gap_max", "gap_std"]
    f = f.join(gg, how="left")
    f["rec_over_gap"] = f.recency_days / (f.gap_mean.fillna(90) + 1)
    # customer x category shares in the last 90 days (cross-entity feature)
    q = past[past.age_d <= 90]
    cs = pd.crosstab(q.customer_id, q.category, normalize="index"); cs.columns = [f"share_{x}" for x in cs.columns]
    f = f.join(cs, how="left").fillna({k: 0 for k in cs.columns})
    out = base.merge(f.reset_index(), on="customer_id", how="left")
    out["tenure_days"] = (T - out.signup_date).dt.days
    out["is_pro"] = (out.plan == "pro").astype(int)
    out["T"] = T
    return out

In [ ]:
c, o = make_data()
Ts_train = pd.to_datetime(["2024-05-01", "2024-06-01", "2024-07-01", "2024-08-01",
                           "2024-09-01", "2024-10-01", "2024-11-01"])
tr = pd.concat([build_features(c, o, T) for T in Ts_train], ignore_index=True)
te = build_features(c, o, "2025-01-01")
print("train rows", len(tr), " test rows", len(te), " churn rates", round(tr.churn.mean(), 3), round(te.churn.mean(), 3))
print(te[["recency_days", "orders_7d", "orders_30d", "orders_90d", "gap_mean", "trend_30_vs_prev60", "rec_over_gap"]]
      .describe().round(2).loc[["mean", "50%", "max"]].to_string())

train rows 16152  test rows 2562  churn rates 0.246 0.254
      recency_days  orders_7d  orders_30d  orders_90d  gap_mean  trend_30_vs_prev60  rec_over_gap
mean         19.35       0.75        3.12        9.17     11.65                1.28          1.83
50%           9.00       0.00        2.00        7.00      8.33                1.00          0.84
max          90.00       8.00       22.00       67.00    153.00               17.00         75.00


In [ ]:
rfm = te[["customer_id", "recency_days", "frequency_all", "monetary_all", "churn"]].copy()
rfm["R"] = pd.qcut(rfm.recency_days, 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["F"] = pd.qcut(rfm.frequency_all.rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M"] = pd.qcut(rfm.monetary_all.rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
def seg(r):
    if r.R >= 4 and r.F >= 4: return "Champions"
    if r.R >= 4: return "Recent"
    if r.R <= 2 and r.F >= 4: return "At risk (were frequent)"
    if r.R <= 2: return "Lapsing"
    return "Middle"
rfm["segment"] = rfm.apply(seg, axis=1)
print(rfm.groupby("segment").churn.agg(["size", "mean"]).round(3).sort_values("mean"))

                         size   mean
segment                             
Champions                 608  0.028
Recent                    510  0.069
Middle                    423  0.121
Lapsing                   770  0.536
At risk (were frequent)   251  0.538


## 6.2 Trends, Lags, Cross-Entity Features, and Do They Help?

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
def auc(cols, seed=0):
    m = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.08, random_state=seed).fit(tr[cols], tr.churn)
    return round(roc_auc_score(te.churn, m.predict_proba(te[cols])[:, 1]), 3)
static = ["is_pro", "tenure_days"]
A = ["recency_days", "orders_90d", "is_pro"]
B = A + ["orders_7d", "orders_30d", "spend_7d", "spend_30d", "spend_90d", "frequency_all", "monetary_all"]
C = B + ["ratio_7_30", "trend_30_vs_prev60", "velocity_spend", "accel_spend", "gap_mean", "gap_max", "gap_std", "rec_over_gap"]
D = C + [x for x in te.columns if x.startswith("share_")] + ["tenure_days", "avg_order_90d"]
print("static only (plan, tenure):", auc(static))
for name, cols in [("A RFM-ish base", A), ("B + windows", B), ("C + trends/gaps", C), ("D + category shares, tenure", D)]:
    print(f"{name:32s} n_feat={len(cols):2d}  test AUC={auc(cols)}")

static only (plan, tenure): 0.612
A RFM-ish base                   n_feat= 3  test AUC=0.891
B + windows                      n_feat=10  test AUC=0.891
C + trends/gaps                  n_feat=18  test AUC=0.887
D + category shares, tenure      n_feat=24  test AUC=0.89


In [ ]:
from sklearn.model_selection import train_test_split
rtr, rte = train_test_split(tr, test_size=0.3, random_state=0)
m = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.08, random_state=0).fit(rtr[D], rtr.churn)
print("random split inside snapshots AUC:", round(roc_auc_score(rte.churn, m.predict_proba(rte[D])[:, 1]), 3))
print("time split AUC:", auc(D))
print("customers in train & test:", len(set(tr.customer_id) & set(te.customer_id)), "of", te.customer_id.nunique())

## 6.3 Feature Selection and Stability

In [ ]:
from sklearn.feature_selection import mutual_info_classif, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
feats = ["recency_days", "frequency_all", "monetary_all", "orders_7d", "orders_30d", "orders_90d",
         "spend_7d", "spend_30d", "spend_90d", "ratio_7_30", "trend_30_vs_prev60", "velocity_spend",
         "accel_spend", "gap_mean", "gap_max", "gap_std", "rec_over_gap", "share_grocery", "share_fashion",
         "share_electronics", "share_home", "tenure_days", "is_pro", "avg_order_90d", "age"]   # 25 columns
Xtr, Xte = tr[feats], te[feats]
cm = Xtr.corr(method="spearman").abs()
print("pairs with |spearman|>0.95:", [(a, b, round(cm.loc[a, b], 3)) for i, a in enumerate(feats) for b in feats[i+1:] if cm.loc[a, b] > 0.95])
mi = pd.Series(mutual_info_classif(Xtr.fillna(0), tr.churn, random_state=0), index=feats).sort_values(ascending=False)
print("MI top 6:", mi.head(6).round(3).to_dict())
Xs = StandardScaler().fit_transform(Xtr.fillna(0))
l1 = LogisticRegression(penalty="l1", solver="liblinear", C=0.05).fit(Xs, tr.churn)
kept = [f for f, w in zip(feats, l1.coef_[0]) if abs(w) > 1e-6]
print("L1 (C=0.05) keeps", len(kept), "of", len(feats), ":", kept)
m = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.08, random_state=0).fit(Xtr, tr.churn)
pi = permutation_importance(m, Xte, te.churn, scoring="roc_auc", n_repeats=5, random_state=0)
print(pd.Series(pi.importances_mean, index=feats).sort_values(ascending=False).head(6).round(3).to_dict())

pairs with |spearman|>0.95: [('frequency_all', 'monetary_all', 0.985), ('orders_7d', 'spend_7d', 0.973), ('orders_30d', 'spend_30d', 0.953), ('orders_90d', 'spend_90d', 0.96)]
MI top 6: {'recency_days': 0.188, 'spend_30d': 0.177, 'orders_30d': 0.173, 'rec_over_gap': 0.145, 'trend_30_vs_prev60': 0.134, 'monetary_all': 0.125}
L1 (C=0.05) keeps 13 of 25 : ['recency_days', 'monetary_all', 'orders_7d', 'orders_30d', 'spend_7d', 'trend_30_vs_prev60', 'accel_spend', 'gap_mean', 'gap_std', 'rec_over_gap', 'share_fashion', 'share_home', 'is_pro']
permutation importance (AUC drop) top 6: {'recency_days': 0.198, 'tenure_days': 0.008, 'rec_over_gap': 0.006, 'is_pro': 0.004, 'gap_mean': 0.003, 'spend_90d': 0.003}
full model AUC: 0.889
top-5 by permutation: ['recency_days', 'tenure_days', 'rec_over_gap', 'is_pro', 'gap_mean']  AUC with only these: 0.894
RFE keeps: ['recency_days', 'orders_7d', 'orders_30d', 'rec_over_gap', 'is_pro']


In [ ]:
pis = pd.Series(pi.importances_mean, index=feats).sort_values(ascending=False)
top5 = list(pis.index[:5]); print("top-5 by permutation:", top5, " AUC with only these:", auc(top5))
rfe = RFE(LogisticRegression(max_iter=2000), n_features_to_select=5).fit(Xs, tr.churn)
print("RFE keeps:", [f for f, k in zip(feats, rfe.support_) if k])

In [ ]:
def psi(expected, actual, bins=10):
    qs = np.unique(np.nanquantile(expected, np.linspace(0, 1, bins + 1)))
    qs[0], qs[-1] = -np.inf, np.inf
    e = np.histogram(expected.dropna(), qs)[0] / expected.notna().sum()
    a = np.histogram(actual.dropna(), qs)[0] / actual.notna().sum()
    e = np.clip(e, 1e-4, None); a = np.clip(a, 1e-4, None)
    return float(np.sum((a - e) * np.log(a / e)))
old = tr[tr["T"] == pd.Timestamp("2024-05-01")]
print({f: round(psi(old[f], te[f]), 3) for f in ["recency_days", "orders_90d", "spend_90d", "tenure_days", "gap_mean"]})
for k in (1.0, 1.2, 1.5, 2.0):
    print("spend x", k, "PSI", round(psi(old.spend_90d, te.spend_90d * k), 3))

{'recency_days': 0.009, 'orders_90d': 0.006, 'spend_90d': 0.02, 'tenure_days': 0.162, 'gap_mean': 0.015}
spend x 1.0 PSI 0.02
spend x 1.2 PSI 0.065
spend x 1.5 PSI 0.191
spend x 2.0 PSI 0.46


## 6.4 Leakage-Safe Pipelines and Training-Serving Skew

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
rng = np.random.default_rng(0)
n, p = 400, 1000
X = rng.normal(size=(n, p)); y = rng.integers(0, 2, n)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
Xk = SelectKBest(f_classif, k=20).fit_transform(X, y)                 # selection uses ALL labels (leak)
leak = cross_val_score(LogisticRegression(max_iter=1000), Xk, y, cv=cv, scoring="roc_auc").mean()
pipe = Pipeline([("sel", SelectKBest(f_classif, k=20)), ("clf", LogisticRegression(max_iter=1000))])
safe = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc").mean()   # selection refit inside each fold
print("pure-noise data: selection outside CV AUC", round(leak, 3), " selection inside pipeline AUC", round(safe, 3))

pure-noise data: selection outside CV AUC 0.757  selection inside pipeline AUC 0.517


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
num = ["recency_days", "orders_30d", "orders_90d", "spend_90d", "gap_mean", "rec_over_gap", "tenure_days"]
cat = ["plan", "channel", "city"]
pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)), ("sc", StandardScaler())]), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
pipe = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000))]).fit(tr[num + cat], tr.churn)
print("pipeline AUC on Jan-25:", round(roc_auc_score(te.churn, pipe.predict_proba(te[num + cat])[:, 1]), 3))
new = te[num + cat].head(2).copy()
new.loc[new.index[0], "city"] = "Jaipur"; new.loc[new.index[1], "gap_mean"] = np.nan
print("scores for odd rows (unseen city, missing gap):", pipe.predict_proba(new)[:, 1].round(3))
print("n transformed columns:", pipe.named_steps["pre"].transform(te[num + cat].head(1)).shape[1])

pipeline AUC on Jan-25: 0.893
scores for odd rows (unseen city, missing gap): [0.102 0.631]
n transformed columns: 19


## Chapter recap

## Practice set

Problem 6.P.1 (easy) An order happened 12 days before T. Which windows (7, 30, 90) contain it? **Answer:** The 30-day and 90-day windows, not the 7-day one.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ds/ch06.html